# 🎓 Veritas AI — Notebook 6: Multi-Teacher Knowledge Distillation & Model Elevation

**Objective**: Synthesize a unified multi-teacher knowledge distillation framework that fuses:
1. **ZeroGPT Teacher**: Token perplexity dynamics, top-k rank predictability, and burstiness shielding.
2. **QuillBot Behavioral Teacher**: 4-class taxonomy distributions (Human, Human+Refined, AI+Refined, Pure AI) with word-level paraphrase sensitivity.
3. **Sequence Ensemble Teacher**: Deep bidirectional contextual representations from RoBERTa (`Hello-SimpleAI/chatgpt-detector-roberta`) and ModernBERT (`rasbt/ai-text-detector-modernbert`).

We distill these complementary signals into an edge-optimized student model (`sentence-transformers/all-MiniLM-L6-v2`), dynamically quantize to INT8 ONNX ($\le 25\text{ MB}$), and benchmark on held-out development splits.

## 1. Mathematical Formulation: Multi-Teacher Fusion & Hinton KD

### Fused Multi-Teacher Soft Target Distribution
For any input chunk $x$ with ground-truth label $y \in \{0, 1, 2, 3\}$, sequence ensemble probability $p_{\text{seq}}(x) = \frac{1}{2}(p_{\text{roberta}} + p_{\text{modernbert}})$, and ZeroGPT predictability $p_{\text{zg}}(x)$:

$$p_{\text{ai}}(x) = \gamma_{\text{seq}} p_{\text{seq}}(x) + (1 - \gamma_{\text{seq}}) p_{\text{zg}}(x)$$

The 4-class soft distribution $\mathbf{q} \in \Delta^3$ is constructed with strict deictic and ESL fairness shielding:
- If human ($y = 0$):
  $$\mathbf{q} = [1 - \epsilon_{\text{polish}}, \epsilon_{\text{polish}}, 0, 0], \quad \epsilon_{\text{polish}} \le 0.05 \text{ (0.00 for ESL)}$$
- If AI attacked / humanized ($y = 2$):
  $$\mathbf{q} = [0.03, 0.12, 0.70 p_{\text{ai}} + 0.15, 0.30(1 - p_{\text{ai}}) + 0.05]$$
- If raw AI ($y = 3$):
  $$\mathbf{q} = [0.02, 0.05, 0.35(1 - p_{\text{ai}}) + 0.05, 0.80 p_{\text{ai}} + 0.10]$$

### Hinton Distillation Objective with Temperature $T$
$$\mathcal{L}_{\text{total}} = \alpha \cdot T^2 \cdot D_{\text{KL}}\left(\sigma\left(\frac{\mathbf{z}_s}{T}\right) \;\Big\|\; \sigma\left(\frac{\mathbf{z}_t}{T}\right)\right) + (1 - \alpha) \cdot \mathcal{L}_{\text{CE}}(\mathbf{z}_s, y)$$

In [ ]:
import os
import sys
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

REPO = os.path.dirname(os.path.abspath('')) if 'notebooks' in os.getcwd() else os.getcwd()
sys.path.insert(0, REPO)

from notebooks.multi_teacher_distillation import MultiTeacherPipeline, build_multi_teacher_training_set
print('PyTorch & Transformers armed. MultiTeacherPipeline imported successfully.')

## 2. Inspecting the Multi-Teacher Ensemble

We instantiate the teacher ensemble combining ZeroGPT causal LM perplexity with RoBERTa and ModernBERT sequence classifiers.

In [ ]:
pipeline = MultiTeacherPipeline(device='cpu', threads=4)

# Diagnostic Test 1: Frontier AI Generation (Claude Opus 5.5 style)
sample_ai = (
    'Furthermore, it is worth noting that machine learning methodologies serve as a testament '
    'to the evolving paradigms in academic discourse, systematically optimizing analytical rigor.'
)
seq_score = pipeline.seq_teacher.score_batch([sample_ai])[0]
zg_res = pipeline.zerogpt_teacher.score_document(sample_ai)
fused_ai = pipeline.compute_fused_target(sample_ai, hard_label=3, seq_p_ai=seq_score, zg_fake_pct=zg_res['fakePercentage'])

print('--- Frontier AI Target ---')
print(f'Seq Ensemble P(AI): {seq_score:.4f}')
print(f'ZeroGPT Fake %:     {zg_res["fakePercentage"]}%')
print(f'Fused 4-Class Target: [Human: {fused_ai[0]:.3f}, Human-Refined: {fused_ai[1]:.3f}, AI-Refined: {fused_ai[2]:.3f}, Pure AI: {fused_ai[3]:.3f}]')

# Diagnostic Test 2: Authentic ESL Human Writing
sample_esl = (
    'My favourite sport is running. When I have free time I usually run in the park '
    'because I feel relaxed after hard working day. My brother taught me how to run fast.'
)
fused_esl = pipeline.compute_fused_target(sample_esl, hard_label=0, is_esl=True)
print('\n--- ESL Human Target (Shielded) ---')
print(f'Fused 4-Class Target: [Human: {fused_esl[0]:.3f}, Human-Refined: {fused_esl[1]:.3f}, AI-Refined: {fused_esl[2]:.3f}, Pure AI: {fused_esl[3]:.3f}]')

## 3. Hyperparameter Exploration: Temperature $T$ and Distillation Weight $\alpha$

We inspect the hyperparameter grid search results logged across $T \in \{1.5, 2.0, 2.5\}$ and $\alpha \in \{0.4, 0.5, 0.6\}$.

In [ ]:
grid_results_path = os.path.join(REPO, 'models', 'candidates', 'multi_teacher_distilled', 'distill_grid_search_results.json')
if os.path.exists(grid_results_path):
    with open(grid_results_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print(f'Optimal Configuration: T = {summary["best_T"]}, alpha = {summary["best_alpha"]} (Validation Loss = {summary["best_val_loss"]:.4f})\n')
    print(f'{"Temperature (T)":<18} | {"Alpha (α)":<12} | {"Val Loss":<12} | {"Time (s)":<10}')
    print('-' * 60)
    for r in summary['grid_results']:
        print(f'{r["T"]:^18.1f} | {r["alpha"]:^12.1f} | {r["val_loss"]:^12.4f} | {r["time_s"]:^10.1f}')
else:
    print('Grid search results will be populated upon run completion.')

## 4. Edge Verification: Dynamic INT8 ONNX Quantization & Parity

Verifying that the quantized model complies with all edge runtime constraints:
- File size $\le 25\text{ MB}$
- Maximum logit error $\le 0.10$
- Pure ONNX Runtime CPU execution (zero PyTorch)

In [ ]:
import onnxruntime as ort

onnx_path = os.path.join(REPO, 'models', 'multi_teacher_distilled', 'student_model_int8.onnx')
if os.path.exists(onnx_path):
    file_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f'INT8 ONNX Model File Size: {file_size_mb:.2f} MB (Budget: <= 25.0 MB)')
    assert file_size_mb <= 25.0, 'Model exceeds edge budget!'
    
    sess_opts = ort.SessionOptions()
    sess_opts.intra_op_num_threads = 2
    session = ort.InferenceSession(onnx_path, sess_opts, providers=['CPUExecutionProvider'])
    print('ONNX Runtime Session successfully initialized on 2 CPU threads.')
else:
    print(f'Model path not found: {onnx_path}')

## 5. Development Split Benchmark Comparison

Comparing `cand:multi_teacher_distilled` against prior checkpoints (`frontier_onnx`, `cand:zerogpt_distilled`, `modernbert`, `hc3_roberta`).

In [ ]:
results_path = os.path.join(REPO, 'data', 'eval', 'results', 'dev_multi_teacher_distilled.json')
if os.path.exists(results_path):
    with open(results_path, 'r', encoding='utf-8') as f:
        eval_data = json.load(f)['results'][0]
    print(f'Detector:                  {eval_data["detector"]}')
    print(f'1% Clean Human FPR Thresh: {eval_data["threshold_1pct_fpr_dev"]:.4f}')
    print(f'Clean Human Realized FPR:  {eval_data["fpr_clean_human"]["rate"]*100:.2f}%')
    print(f'ESL Learner Realized FPR:  {eval_data["fpr_esl"]["rate"]*100:.2f}%')
    print(f'Overall AUROC:             {eval_data.get("auroc_pooled", 0.0):.4f}')
else:
    print('Evaluation results awaiting dev split scoring pass.')